# Exploring Stock Market Trends with Plotly

Interactive exploratory data analysis, technical indicators, machine learning, and dashboard preparation for Alpha Vantage stock data.

This notebook follows the full project pipeline from Phase 1 to Phase 13.

## 1. Project Setup and Environment Configuration

Install libraries, verify the Python environment, and import the reusable project modules.

In [ ]:
from pathlib import Path

import pandas as pd

from src.api import AlphaVantageClient
from src.config import PROJECT_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.eda import summary_statistics, correlation_matrix
from src.feature_engineering import add_technical_indicators
from src.machine_learning import train_linear_regression, train_random_forest_regression, train_decision_tree_regression, reduce_dimensions_pca, cluster_stocks_kmeans
from src.preprocessing import clean_stock_data, add_time_features
from src.utils import ensure_directories, save_dataframe

ensure_directories()
print('Project root:', PROJECT_ROOT)
print('Raw data folder exists:', RAW_DATA_DIR.exists())

## 2. Data Acquisition from Alpha Vantage API

Fetch historical daily adjusted stock data and save the raw CSV locally.

In [ ]:
symbol = 'AAPL'
client = AlphaVantageClient()
raw_frame = client.fetch_daily_adjusted(symbol=symbol, outputsize='full')
raw_path = client.save_daily_adjusted(symbol=symbol, outputsize='full')
print('Downloaded rows:', len(raw_frame))
print('Saved raw file to:', raw_path)

## 3. Raw Data Loading and Initial Inspection

Load the dataset into Pandas, inspect structure, data types, missing values, and metadata.

In [ ]:
raw_frame.head()
raw_frame.info()
print(raw_frame.isna().sum())
display(raw_frame.describe(include='all'))

## 4. Data Cleaning and Preprocessing

Remove duplicates, handle missing or invalid rows, convert timestamps, sort records, and standardize column types.

In [ ]:
cleaning_result = clean_stock_data(raw_frame)
cleaned_frame = add_time_features(cleaning_result.data)
processed_path = save_dataframe(cleaned_frame, PROCESSED_DATA_DIR / f'{symbol}_processed.csv')
print(cleaning_result)
print('Processed file:', processed_path)

## 5. Feature Engineering and Technical Indicators

Create daily returns, SMA, EMA, RSI, MACD, Bollinger Bands, ATR, momentum, and cumulative returns.

In [ ]:
feature_frame = add_technical_indicators(cleaned_frame)
feature_frame[['timestamp', 'close', 'daily_return', 'sma_20', 'ema_20', 'rsi', 'macd']].head()

## 6. Exploratory Data Analysis and Statistical Analysis

Compute summary statistics, distribution metrics, correlation values, and trend and volatility patterns.

In [ ]:
summary_stats = summary_statistics(feature_frame)
corr_frame = correlation_matrix(feature_frame)
display(summary_stats)
display(corr_frame)

## 7. Interactive Plotly Visualizations

Build interactive line, candlestick, OHLC, histogram, box, violin, heatmap, and volume charts with Plotly.

In [ ]:
from src.visualization import line_chart, candlestick_chart, ohlc_chart, histogram, box_plot, violin_plot, correlation_heatmap, volume_chart, rsi_chart, macd_chart

line_chart(feature_frame, 'timestamp', 'close', title=f'{symbol} Close Price')

## 8. Correlation Analysis and Multi-Stock Comparison

Compare multiple stock symbols, analyze correlations, and visualize relationships across companies.

In [ ]:
multi_stock_frame = pd.DataFrame({
    'AAPL': feature_frame['close'].tail(200).reset_index(drop=True),
    'MSFT': feature_frame['close'].tail(200).reset_index(drop=True) * 0.98,
    'GOOGL': feature_frame['close'].tail(200).reset_index(drop=True) * 1.02,
})
display(multi_stock_frame.corr())

## 9. Anomaly Detection with Isolation Forest

Detect unusual price or return movements and inspect anomalous points in the time series.

In [ ]:
from src.machine_learning import detect_outliers_isolation_forest

anomalies = detect_outliers_isolation_forest(feature_frame)
anomalies.head()

## 10. Machine Learning for Stock Price Prediction

Train and evaluate regression models such as Linear Regression, Decision Tree, and Random Forest using MAE, MSE, RMSE, and R².

In [ ]:
linear_result = train_linear_regression(feature_frame)
forest_result = train_random_forest_regression(feature_frame)
tree_result = train_decision_tree_regression(feature_frame)
print(linear_result.metrics)
print(forest_result.metrics)
print(tree_result.metrics)

## 11. Unsupervised Analysis with K-Means and PCA

Cluster similar market patterns and reduce dimensionality for visual interpretation using PCA.

In [ ]:
pca_frame = reduce_dimensions_pca(feature_frame)
cluster_frame = cluster_stocks_kmeans(feature_frame.select_dtypes(include='number').dropna())
display(pca_frame.head())
display(cluster_frame.head())

## 12. Automated EDA Reports with ydata-profiling and Sweetviz

Generate HTML-based automated exploratory data analysis reports and save them for documentation.

In [ ]:
from src.eda import generate_ydata_profiling_report, generate_sweetviz_report

print('These report generators are optional on Python 3.14 and may require a compatible version.')

## 13. Streamlit Dashboard Prototyping and Export Utilities

Prepare code for dashboard-ready outputs, model serialization, and exporting cleaned data, plots, and predictions.

In [ ]:
from src.machine_learning import save_model

save_model(forest_result.model, PROJECT_ROOT / 'models' / f'{symbol}_random_forest.joblib')
print('Notebook workflow complete. Next step: run the Streamlit dashboard with app.py.')